In [ ]:
import folium
import geojson
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import plotly.express as px
import plotly.graph_objects as go

from folium import plugins
from folium.plugins import HeatMap
from matplotlib.ticker import FuncFormatter, MultipleLocator
from scipy import stats

In [ ]:
df_rent = pd.read_csv('data/mean_rents.csv')
df_rent['Borough'] = df_rent['Borough'].str.upper()
df_rent.head()

In [ ]:
df_rent['apartmentType'].unique()

In [ ]:
df_income = pd.read_csv('data/income_data.csv')
df_income['Work Location Borough'] = df_income['Work Location Borough'].replace('RICHMOND', 'STATEN ISLAND')
df_income.head()

In [ ]:
df_income['Total Income'] = df_income['Regular Gross Paid'] + df_income['Total OT Paid'] + df_income['Total Other Pay']

In [ ]:
699.41+95897.0	

In [ ]:
df_income['Pay Basis'].unique()

In [ ]:
df_income[df_income['Last Name']=='BRAY']

In [ ]:
df_rent['apartmentType'].unique()

In [ ]:
df_income_borough_year = df_income.groupby(['Fiscal Year', 'Work Location Borough'])['Total Income'].mean().reset_index()
df_income_borough_year.columns = ['Fiscal Year', 'Borough', 'Average Income']
df_income_borough_year.head()

In [ ]:
df_income_borough_year_pivot = df_income.pivot_table(
    index='Work Location Borough', 
    columns='Fiscal Year', 
    values='Total Income', 
    aggfunc='mean'
)
df_income_borough_year_pivot.reset_index(inplace=True)
df_income_borough_year_pivot.head()

In [ ]:
# Convert annual income to monthly
df_income_monthly = df_income.pivot_table(
    index='Work Location Borough',
    columns='Fiscal Year',
    values='Total Income',
    aggfunc='mean'
) / 12



In [ ]:
import json
import requests

# Load from URL
url = "https://raw.githubusercontent.com/dwillis/nyc-maps/master/boroughs.geojson"
counties = requests.get(url).json()

# Extract one-bedroom rent data for January 2016
df_onebd_rent = df_rent[df_rent['apartmentType'] == 'One bedroom'].copy()
df_data = df_onebd_rent[['Borough', '2016-01']].copy()
df_data.columns = ['Borough', 'Rent_Jan_2016']
df_data = df_data.dropna()

# Get 2016 income and convert to monthly
df_income_2016 = df_income_borough_year_pivot[['Work Location Borough', 2016]].copy()
df_income_2016.columns = ['Borough', 'Annual_Income']
df_income_2016['Monthly_Income'] = df_income_2016['Annual_Income'] / 12

df_data = df_data.merge(df_income_2016[['Borough', 'Monthly_Income']], on='Borough')


In [ ]:
df_onebd_rent

In [ ]:
df_income['Work Location Borough'].unique()

In [ ]:
temp = df_income[df_income['Work Location Borough']=='MANHATTAN']
temp[temp['Fiscal Year']==2016]['Total Income'].mean()

In [ ]:
62492/12

In [ ]:
def calculate_after_tax_income(annual_income, borough):
    """
    Calculate after-tax income for 2016 in NYC
    Includes: Federal, NY State, NYC (if applicable), and FICA taxes
    """
    
    # FICA taxes (Social Security + Medicare) - flat rate
    fica_rate = 0.0765
    fica_tax = annual_income * fica_rate
    
    # Federal income tax (2016 brackets - single filer)
    federal_tax = 0
    if annual_income > 91150:
        federal_tax += (annual_income - 91150) * 0.28
        federal_tax += (91150 - 37650) * 0.25
        federal_tax += 37650 * 0.15
    elif annual_income > 37650:
        federal_tax += (annual_income - 37650) * 0.25
        federal_tax += 37650 * 0.15
    else:
        federal_tax = annual_income * 0.10
    
    # NY State income tax (2016 brackets - single filer)
    ny_state_tax = 0
    if annual_income > 80650:
        ny_state_tax += (annual_income - 80650) * 0.0685
        ny_state_tax += (80650 - 21400) * 0.065
        ny_state_tax += 21400 * 0.04
    elif annual_income > 21400:
        ny_state_tax += (annual_income - 21400) * 0.065
        ny_state_tax += 21400 * 0.04
    else:
        ny_state_tax = annual_income * 0.04
    
    # NYC income tax (applies to all 5 boroughs)
    nyc_tax = 0
    if annual_income > 90000:
        nyc_tax += (annual_income - 90000) * 0.04
        nyc_tax += (90000 - 50000) * 0.038
        nyc_tax += 50000 * 0.035
    elif annual_income > 50000:
        nyc_tax += (annual_income - 50000) * 0.038
        nyc_tax += 50000 * 0.035
    else:
        nyc_tax = annual_income * 0.035
    
    total_tax = fica_tax + federal_tax + ny_state_tax + nyc_tax
    after_tax = annual_income - total_tax
    
    return after_tax

# Apply to monthly income
df_data['Annual_Income'] = df_data['Monthly_Income'] * 12
df_data['After_Tax_Annual'] = df_data.apply(
    lambda row: calculate_after_tax_income(row['Annual_Income'], row['Borough']), 
    axis=1
)
df_data['After_Tax_Monthly'] = df_data['After_Tax_Annual'] / 12

# Calculate difference using after-tax income
df_data['Difference'] = df_data['After_Tax_Monthly'] - df_data['Rent_Jan_2016']

In [ ]:
df_data

In [ ]:
df_data['Borough'] = df_data['Borough'].str.title()

fig = px.choropleth_map(
    df_data,
    geojson=counties,
    locations='Borough',
    color='Difference',
    featureidkey='properties.BoroName',
    hover_name='Borough',
    hover_data={
        'Difference': ':.2f',
        'Monthly_Income': ':.2f', 
        'Rent_Jan_2016': ':.2f'
    },
    color_continuous_scale='RdYlGn',
    range_color=[df_data['Difference'].min(), df_data['Difference'].max()],
    labels={'Difference': 'Income minus Rent ($)'},
    title='Income vs Rent Difference - One Bedroom (January 2016)',
    zoom=9, center={'lat': 40.7128, 'lon': -73.9000},
    opacity=0.7,
)

fig.update_layout(
    margin={'r': 0, 't': 0, 'l': 0, 'b': 0}
)

fig.show()

In [ ]:
def calculate_after_tax_income(annual_income, borough, year):
    """
    Calculate after-tax income for a given year in NYC
    Includes: Federal, NY State, NYC, and FICA taxes
    """
    
    # FICA taxes - flat rate (unchanged 2014-2025)
    fica_rate = 0.0765
    fica_tax = annual_income * fica_rate
    
    # Federal income tax brackets (single filer) - varies by year
    federal_brackets = {
        2014: [(9075, 0.10), (36900, 0.15), (89075, 0.25), (189300, 0.28), (411500, 0.33), (413200, 0.35), (float('inf'), 0.396)],
        2015: [(9225, 0.10), (37450, 0.15), (90750, 0.25), (189300, 0.28), (411500, 0.33), (413200, 0.35), (float('inf'), 0.396)],
        2016: [(9300, 0.10), (37650, 0.15), (91150, 0.25), (190150, 0.28), (413350, 0.33), (415050, 0.35), (float('inf'), 0.396)],
        2017: [(9325, 0.10), (37950, 0.15), (91900, 0.25), (191650, 0.28), (416700, 0.33), (418400, 0.35), (float('inf'), 0.396)],
        2018: [(9525, 0.10), (38700, 0.12), (82500, 0.22), (157500, 0.24), (200000, 0.32), (500000, 0.35), (float('inf'), 0.37)],
        2019: [(9700, 0.10), (39475, 0.12), (84200, 0.22), (160725, 0.24), (204100, 0.32), (511000, 0.35), (float('inf'), 0.37)],
        2020: [(9875, 0.10), (40125, 0.12), (85525, 0.22), (163300, 0.24), (207350, 0.32), (518400, 0.35), (float('inf'), 0.37)],
        2021: [(9950, 0.10), (40525, 0.12), (86375, 0.22), (164925, 0.24), (209425, 0.32), (523600, 0.35), (float('inf'), 0.37)],
        2022: [(10275, 0.10), (41775, 0.12), (89075, 0.22), (170050, 0.24), (215950, 0.32), (539900, 0.35), (float('inf'), 0.37)],
        2023: [(11000, 0.10), (44725, 0.12), (95375, 0.22), (182100, 0.24), (231250, 0.32), (578125, 0.35), (float('inf'), 0.37)],
        2024: [(11600, 0.10), (47150, 0.12), (100525, 0.22), (191950, 0.24), (243725, 0.32), (609350, 0.35), (float('inf'), 0.37)],
        2025: [(11950, 0.10), (48575, 0.12), (103500, 0.22), (198050, 0.24), (250525, 0.32), (626350, 0.35), (float('inf'), 0.37)],
    }
    
    # Calculate federal tax
    federal_tax = 0
    brackets = federal_brackets.get(year, federal_brackets[2016])
    previous_limit = 0
    for limit, rate in brackets:
        if annual_income > previous_limit:
            taxable_in_bracket = min(annual_income, limit) - previous_limit
            federal_tax += taxable_in_bracket * rate
            previous_limit = limit
        else:
            break
    
    # NY State and NYC taxes (simplified - use 2016 rates for all years)
    ny_state_tax = 0
    if annual_income > 80650:
        ny_state_tax += (annual_income - 80650) * 0.0685
        ny_state_tax += (80650 - 21400) * 0.065
        ny_state_tax += 21400 * 0.04
    elif annual_income > 21400:
        ny_state_tax += (annual_income - 21400) * 0.065
        ny_state_tax += 21400 * 0.04
    else:
        ny_state_tax = annual_income * 0.04
    
    nyc_tax = 0
    if annual_income > 90000:
        nyc_tax += (annual_income - 90000) * 0.04
        nyc_tax += (90000 - 50000) * 0.038
        nyc_tax += 50000 * 0.035
    elif annual_income > 50000:
        nyc_tax += (annual_income - 50000) * 0.038
        nyc_tax += 50000 * 0.035
    else:
        nyc_tax = annual_income * 0.035
    
    total_tax = fica_tax + federal_tax + ny_state_tax + nyc_tax
    after_tax = annual_income - total_tax
    
    return after_tax

In [ ]:
# Get all months from the rent data
months = [col for col in df_rent.columns if col.startswith('20') and '-' in col]

# Create a list to store data for each month
data_frames = []

for month in months:
    # Extract rent data for this month
    df_onebd_rent = df_rent[df_rent['apartmentType'] == 'One bedroom'].copy()
    df_month = df_onebd_rent[['Borough', month]].copy()
    df_month.columns = ['Borough', 'Rent']
    df_month = df_month.dropna()
    
    # Extract year from month column (e.g., '2016-01' -> 2016)
    year = int(month.split('-')[0])
    
    # Get income for that year and convert to monthly
    df_income_year = df_income_borough_year_pivot[['Work Location Borough', year]].copy()
    df_income_year.columns = ['Borough', 'Annual_Income']
    df_income_year['Monthly_Income'] = df_income_year['Annual_Income'] / 12
    
    # Merge
    df_month = df_month.merge(df_income_year[['Borough', 'Monthly_Income']], on='Borough')
    
    # Calculate after-tax income and difference
    df_month['Annual_Income'] = df_month['Monthly_Income'] * 12
    df_month['After_Tax_Annual'] = df_month.apply(
        lambda row: calculate_after_tax_income(row['Annual_Income'], row['Borough'], year), 
        axis=1
    )
    df_month['After_Tax_Monthly'] = df_month['After_Tax_Annual'] / 12
    df_month['Difference'] = df_month['After_Tax_Monthly'] - df_month['Rent']
    df_month['Month'] = month
    df_month['Borough'] = df_month['Borough'].str.title()
    
    data_frames.append(df_month)

In [ ]:


# Combine all months
df_animated = pd.concat(data_frames, ignore_index=True)

# Get global min/max for consistent color scale
global_min = df_animated['Difference'].min()
global_max = df_animated['Difference'].max()

# Create animated choropleth
fig = px.choropleth_map(
    df_animated,
    geojson=counties,
    locations='Borough',
    color='Difference',
    featureidkey='properties.BoroName',
    hover_name='Borough',
    hover_data={'Difference': ':.2f', 'Monthly_Income': ':.2f', 'Rent': ':.2f'},
    color_continuous_scale='RdYlGn',
    range_color=[global_min, global_max],
    labels={'Difference': 'Income minus Rent ($)'},
    title='Income vs Rent Difference - One Bedroom (2014-2025)',
    animation_frame='Month',
    opacity=0.7,
    zoom=9, center={'lat': 40.7128, 'lon': -73.9000}
)

fig.update_layout(
    margin={'r': 0, 't': 0, 'l': 0, 'b': 0}
)

fig.show()

In [ ]:
# Get all years available
years = sorted(df_income_borough_year_pivot.columns[1:])  # Skip the borough column

# Create a list to store data for each year
data_frames = []

for year in years:
    # Get average monthly rent for this year (average across all 12 months)
    rent_cols = [col for col in df_rent.columns if col.startswith(f'{year}-')]
    df_onebd_rent = df_rent[df_rent['apartmentType'] == 'One bedroom'].copy()
    df_year = df_onebd_rent[['Borough']].copy()
    df_year['Rent'] = df_onebd_rent[rent_cols].mean(axis=1)
    df_year = df_year.dropna()
    
    # Get income for that year and convert to monthly
    df_income_year = df_income_borough_year_pivot[['Work Location Borough', year]].copy()
    df_income_year.columns = ['Borough', 'Annual_Income']
    df_income_year['Monthly_Income'] = df_income_year['Annual_Income'] / 12
    
    # Merge
    df_year = df_year.merge(df_income_year[['Borough', 'Monthly_Income']], on='Borough')
    
    # Calculate after-tax income and difference
    df_year['Annual_Income'] = df_year['Monthly_Income'] * 12
    df_year['After_Tax_Annual'] = df_year.apply(
        lambda row: calculate_after_tax_income(row['Annual_Income'], row['Borough'], year), 
        axis=1
    )
    df_year['After_Tax_Monthly'] = df_year['After_Tax_Annual'] / 12
    df_year['Difference'] = df_year['After_Tax_Monthly'] - df_year['Rent']
    df_year['Year'] = str(year)
    df_year['Borough'] = df_year['Borough'].str.title()
    
    data_frames.append(df_year)

# Combine all years
df_animated = pd.concat(data_frames, ignore_index=True)

# Get global min/max for consistent color scale
global_min = df_animated['Difference'].min()
global_max = df_animated['Difference'].max()

# Create animated choropleth
fig = px.choropleth_map(
    df_animated,
    geojson=counties,
    locations='Borough',
    color='Difference',
    featureidkey='properties.BoroName',
    hover_name='Borough',
    hover_data={'Difference': ':.2f', 'Monthly_Income': ':.2f', 'Rent': ':.2f'},
    color_continuous_scale='RdYlGn',
    range_color=[global_min, global_max],
    labels={'Difference': 'Income minus Rent ($)'},
    title='Average Yearly Income vs Rent - One Bedroom',
    animation_frame='Year',
    opacity=0.7,
    zoom=9, center={'lat': 40.7128, 'lon': -73.9000}
)

fig.update_layout(
    margin={'r': 0, 't': 0, 'l': 0, 'b': 0}
)

fig.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

apartment_types = ['Studio', 'One bedroom', 'Two bedrooms', 'Three+ bedrooms']
years = sorted(df_income_borough_year_pivot.columns[1:])

# Get global min/max
all_differences = []
for year in years:
    for apt_type in apartment_types:
        rent_cols = [col for col in df_rent.columns if col.startswith(f'{year}-')]
        df_apt_rent = df_rent[df_rent['apartmentType'] == apt_type].copy()
        df_apt = df_apt_rent[['Borough']].copy()
        df_apt['Rent'] = df_apt_rent[rent_cols].mean(axis=1)
        df_apt = df_apt.dropna()
        
        if len(df_apt) > 0:
            df_income_year = df_income_borough_year_pivot[['Work Location Borough', year]].copy()
            df_income_year.columns = ['Borough', 'Annual_Income']
            df_income_year['Monthly_Income'] = df_income_year['Annual_Income'] / 12
            
            df_apt = df_apt.merge(df_income_year[['Borough', 'Monthly_Income']], on='Borough')
            df_apt['Annual_Income'] = df_apt['Monthly_Income'] * 12
            df_apt['After_Tax_Annual'] = df_apt.apply(
                lambda r: calculate_after_tax_income(r['Annual_Income'], r['Borough'], year), 
                axis=1
            )
            df_apt['After_Tax_Monthly'] = df_apt['After_Tax_Annual'] / 12
            df_apt['Difference'] = df_apt['After_Tax_Monthly'] - df_apt['Rent']
            all_differences.extend(df_apt['Difference'].dropna().values)

global_min = min(all_differences)
global_max = max(all_differences)



In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

apartment_types = ['Studio', 'One bedroom', 'Two bedrooms', 'Three+ bedrooms']
years = sorted(df_income_borough_year_pivot.columns[1:])

# Prepare data for all combinations to get global min/max
all_differences = []
for year in years:
    for apt_type in apartment_types:
        rent_cols = [col for col in df_rent.columns if col.startswith(f'{year}-')]
        df_apt_rent = df_rent[df_rent['apartmentType'] == apt_type].copy()
        df_apt = df_apt_rent[['Borough']].copy()
        df_apt['Rent'] = df_apt_rent[rent_cols].mean(axis=1)
        df_apt = df_apt.dropna()
        
        if len(df_apt) > 0:
            df_income_year = df_income_borough_year_pivot[['Work Location Borough', year]].copy()
            df_income_year.columns = ['Borough', 'Annual_Income']
            df_income_year['Monthly_Income'] = df_income_year['Annual_Income'] / 12
            
            df_apt = df_apt.merge(df_income_year[['Borough', 'Monthly_Income']], on='Borough')
            df_apt['Annual_Income'] = df_apt['Monthly_Income'] * 12
            df_apt['After_Tax_Annual'] = df_apt.apply(
                lambda r: calculate_after_tax_income(r['Annual_Income'], r['Borough'], year), 
                axis=1
            )
            df_apt['After_Tax_Monthly'] = df_apt['After_Tax_Annual'] / 12
            df_apt['Difference'] = df_apt['After_Tax_Monthly'] - df_apt['Rent']
            all_differences.extend(df_apt['Difference'].dropna().values)

global_min = min(all_differences)
global_max = max(all_differences)



In [ ]:
df_apt

In [ ]:
# Create subplots: 2x2 grid
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=apartment_types,
    specs=[[{'type': 'mapbox'}, {'type': 'mapbox'}],
           [{'type': 'mapbox'}, {'type': 'mapbox'}]]
)

# Create frames for animation
frames = []
for year in years:
    frame_traces = []
    for apt_type in apartment_types:
        rent_cols = [col for col in df_rent.columns if col.startswith(f'{year}-')]
        df_apt_rent = df_rent[df_rent['apartmentType'] == apt_type].copy()
        df_apt = df_apt_rent[['Borough']].copy()
        df_apt['Rent'] = df_apt_rent[rent_cols].mean(axis=1)
        df_apt = df_apt.dropna()
        
        if len(df_apt) > 0:
            df_income_year = df_income_borough_year_pivot[['Work Location Borough', year]].copy()
            df_income_year.columns = ['Borough', 'Annual_Income']
            df_income_year['Monthly_Income'] = df_income_year['Annual_Income'] / 12
            
            df_apt = df_apt.merge(df_income_year[['Borough', 'Monthly_Income']], on='Borough')
            df_apt['Annual_Income'] = df_apt['Monthly_Income'] * 12
            df_apt['After_Tax_Annual'] = df_apt.apply(
                lambda r: calculate_after_tax_income(r['Annual_Income'], r['Borough'], year), 
                axis=1
            )
            df_apt['After_Tax_Monthly'] = df_apt['After_Tax_Annual'] / 12
            df_apt['Difference'] = df_apt['After_Tax_Monthly'] - df_apt['Rent']
            df_apt['Borough'] = df_apt['Borough'].str.title()
        else:
            df_apt = pd.DataFrame({'Borough': [], 'Difference': []})
        
        trace = go.Choroplethmapbox(
            geojson=counties,
            locations=df_apt['Borough'],
            z=df_apt['Difference'],
            featureidkey='properties.BoroName',
            colorscale='RdYlGn',
            zmin=global_min,
            zmax=global_max,
            showscale=(len(frame_traces) == 0),
            hovertemplate='<b>%{location}</b><br>Difference: $%{z:.2f}<extra></extra>',
        )
        frame_traces.append(trace)
    
    frames.append(go.Frame(data=frame_traces, name=str(year)))

# Add initial data for first year
year = years[0]
for apt_type in apartment_types:
    rent_cols = [col for col in df_rent.columns if col.startswith(f'{year}-')]
    df_apt_rent = df_rent[df_rent['apartmentType'] == apt_type].copy()
    df_apt = df_apt_rent[['Borough']].copy()
    df_apt['Rent'] = df_apt_rent[rent_cols].mean(axis=1)
    df_apt = df_apt.dropna()
    
    if len(df_apt) > 0:
        df_income_year = df_income_borough_year_pivot[['Work Location Borough', year]].copy()
        df_income_year.columns = ['Borough', 'Annual_Income']
        df_income_year['Monthly_Income'] = df_income_year['Annual_Income'] / 12
        
        df_apt = df_apt.merge(df_income_year[['Borough', 'Monthly_Income']], on='Borough')
        df_apt['Annual_Income'] = df_apt['Monthly_Income'] * 12
        df_apt['After_Tax_Annual'] = df_apt.apply(
            lambda r: calculate_after_tax_income(r['Annual_Income'], r['Borough'], year), 
            axis=1
        )
        df_apt['After_Tax_Monthly'] = df_apt['After_Tax_Annual'] / 12
        df_apt['Difference'] = df_apt['After_Tax_Monthly'] - df_apt['Rent']
        df_apt['Borough'] = df_apt['Borough'].str.title()
    else:
        df_apt = pd.DataFrame({'Borough': [], 'Difference': []})
    
    row = apartment_types.index(apt_type) // 2 + 1
    col = apartment_types.index(apt_type) % 2 + 1
    
    fig.add_trace(
        go.Choroplethmapbox(
            geojson=counties,
            locations=df_apt['Borough'],
            z=df_apt['Difference'],
            featureidkey='properties.BoroName',
            colorscale='RdYlGn',
            zmin=global_min,
            zmax=global_max,
            showscale=(apartment_types.index(apt_type) == 0),
            hovertemplate='<b>%{location}</b><br>Difference: $%{z:.2f}<extra></extra>'
        ),
        row=row, col=col
    )

fig.frames = frames

fig.update_layout(
    mapbox=dict(style='carto-positron', zoom=8, center={'lat': 40.7128, 'lon': -73.9000}),
    mapbox2=dict(style='carto-positron', zoom=8, center={'lat': 40.7128, 'lon': -73.9000}),
    mapbox3=dict(style='carto-positron', zoom=8, center={'lat': 40.7128, 'lon': -73.9000}),
    mapbox4=dict(style='carto-positron', zoom=8, center={'lat': 40.7128, 'lon': -73.9000}),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        buttons=[
            dict(label='Play', method='animate',
                 args=[None, {'frame': {'duration': 1500, 'redraw': True}, 'fromcurrent': True}]),
            dict(label='Pause', method='animate',
                 args=[[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate'}])
        ]
    )],
    sliders=[dict(
        active=0,
        steps=[dict(args=[[str(year)], dict(frame={'duration': 1000, 'redraw': True}, mode='immediate')],
                   label=str(year)) for year in years]
    )],
    title_text='Income vs Rent Difference - All Apartment Types (2014-2025)',
    height=700,
    margin={'r': 0, 't': 50, 'l': 0, 'b': 0}
)

fig.show()
